# 02 - cleaning the trip data

we clean each cab type separately before combining them. the goal is a consistent schema across yellow, green and hvfhv so we can stack them later.

for each type we:
- drop columns we don't need (payment info, vendor ids etc.)
- compute trip duration, speed and price per mile
- drop unknown pickup/dropoff zones (264 and 265 are unassigned)
- floor timestamps to 30 minute buckets
- flag outliers using iqr rather than dropping them, so we can inspect what got flagged

## yellow cabs

yellow cabs are metered taxis and have the most complete trip info. the raw files use `tpep_pickup_datetime` and `tpep_dropoff_datetime` for timestamps.

In [ ]:
import pandas as pd

parquet_file = r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data/yellow.parquet'

df = pd.read_parquet(parquet_file, engine='auto')

cols_to_drop = [
    'passenger_count', 'store_and_fwd_flag', 'payment_type',
    'mta_tax', 'improvement_surcharge', 'extra', 'VendorID',
    'RatecodeID', 'tolls_amount', 'congestion_surcharge',
    'tip_amount', 'airport_fee', 'total_amount', 'cbd_congestion_fee'
]
df = df.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
# compute derived columns
pickup  = pd.to_datetime(df['tpep_pickup_datetime'])
dropoff = pd.to_datetime(df['tpep_dropoff_datetime'])
df['trip_duration_min'] = (dropoff - pickup).dt.total_seconds() / 60

df['price_per_mi'] = df['fare_amount'] / df['trip_distance']
df['speed_mph']    = df['trip_distance'] / (df['trip_duration_min'] / 60)

df['is_shared'] = None
df['cab_type']  = 'yellow'

df.columns = ['pickup_datetime', 'dropoff_datetime', 'trip_distance_mi',
              'pu_zone_id', 'do_zone_id', 'fare', 'trip_duration_min',
              'price_per_mi', 'speed_mph', 'is_shared', 'cab_type']

df = df[['pickup_datetime', 'dropoff_datetime',
         'pu_zone_id', 'do_zone_id', 'cab_type', 'trip_distance_mi',
         'trip_duration_min', 'speed_mph', 'fare', 'is_shared', 'price_per_mi']]

In [ ]:
# drop unknown zones
df = df[~df['pu_zone_id'].isin([264, 265])]
df = df[~df['do_zone_id'].isin([264, 265])]

df = df[df['trip_duration_min'] > 0]
df = df.sort_values(['pu_zone_id', 'pickup_datetime'])

# floor to 30 min buckets
df['pickup_datetime']  = df['pickup_datetime'].dt.floor('30min')
df['dropoff_datetime'] = df['dropoff_datetime'].dt.floor('30min')

# keep trips with non-zero distance and duration
df = df[(df['trip_distance_mi'] > 0) & (df['trip_duration_min'] > 0)]

In [ ]:
# flag outliers using iqr - 1.5x for distance, 3x for fare/price/speed
dist_q1, dist_q3 = df['trip_distance_mi'].quantile([0.25, 0.75])
dist_iqr = dist_q3 - dist_q1
dist_lo  = dist_q1 - 1.5 * dist_iqr
dist_hi  = dist_q3 + 1.5 * dist_iqr
df['is_dist_outlier'] = (df['trip_distance_mi'] < dist_lo) | (df['trip_distance_mi'] > dist_hi)

ppm_q1, ppm_q3 = df['price_per_mi'].quantile([0.25, 0.75])
ppm_iqr = ppm_q3 - ppm_q1
ppm_lo  = ppm_q1 - 3 * ppm_iqr
ppm_hi  = ppm_q3 + 3 * ppm_iqr
df['is_pricepmi_outlier'] = (df['price_per_mi'] < ppm_lo) | (df['price_per_mi'] > ppm_hi)

fare_q1, fare_q3 = df['fare'].quantile([0.25, 0.75])
fare_iqr = fare_q3 - fare_q1
fare_lo  = fare_q1 - 3 * fare_iqr
fare_hi  = fare_q3 + 3 * fare_iqr
df['is_fare_outlier'] = (df['fare'] < fare_lo) | (df['fare'] > fare_hi)

df['is_speed_outlier'] = df['speed_mph'] > 100

# check how many same-zone trips also have outlier price per mile
zero_count = ((df['is_pricepmi_outlier'] == True) & (df['pu_zone_id'] == df['do_zone_id'])).sum()
print(zero_count)
print((df['speed_mph'] > 100).sum())

df.to_parquet(r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data/clean_yellow.parquet', index=False)

## green cabs

green cabs are the outer-borough equivalent of yellows. same schema, different timestamp column names (`lpep_*`). we apply exactly the same cleaning steps.

In [ ]:
import pandas as pd

parquet_file = r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data/green.parquet'

df = pd.read_parquet(parquet_file, engine='auto')

cols_to_drop = [
    'passenger_count', 'store_and_fwd_flag', 'RatecodeID', 'payment_type', 'trip_type',
    'improvement_surcharge', 'mta_tax', 'extra', 'VendorID',
    'tolls_amount', 'congestion_surcharge', 'ehail_fee',
    'tip_amount', 'airport_fee', 'total_amount', 'cbd_congestion_fee'
]
df = df.drop(columns=cols_to_drop, errors='ignore')

pickup  = pd.to_datetime(df['lpep_pickup_datetime'])
dropoff = pd.to_datetime(df['lpep_dropoff_datetime'])
df['trip_duration_min'] = (dropoff - pickup).dt.total_seconds() / 60

df['price_per_mi'] = df['fare_amount'] / df['trip_distance']
df['speed_mph']    = df['trip_distance'] / (df['trip_duration_min'] / 60)

df['is_shared'] = None
df['cab_type']  = 'green'

df.columns = ['pickup_datetime', 'dropoff_datetime', 'pu_zone_id',
              'do_zone_id', 'trip_distance_mi', 'fare', 'trip_duration_min',
              'price_per_mi', 'speed_mph', 'is_shared', 'cab_type']

df = df[['pickup_datetime', 'dropoff_datetime',
         'pu_zone_id', 'do_zone_id', 'cab_type', 'trip_distance_mi',
         'trip_duration_min', 'speed_mph', 'fare', 'is_shared', 'price_per_mi']]

In [ ]:
df = df[~df['pu_zone_id'].isin([264, 265])]
df = df[~df['do_zone_id'].isin([264, 265])]
df = df[df['trip_duration_min'] > 0]
df = df.sort_values(['pu_zone_id', 'pickup_datetime'])

df['pickup_datetime']  = df['pickup_datetime'].dt.floor('30min')
df['dropoff_datetime'] = df['dropoff_datetime'].dt.floor('30min')

df = df[(df['trip_distance_mi'] > 0) & (df['trip_duration_min'] > 0)].copy()

dist_q1, dist_q3 = df['trip_distance_mi'].quantile([0.25, 0.75])
dist_iqr = dist_q3 - dist_q1
dist_lo  = dist_q1 - 3.0 * dist_iqr
dist_hi  = dist_q3 + 3.0 * dist_iqr
df['is_dist_outlier'] = (df['trip_distance_mi'] < dist_lo) | (df['trip_distance_mi'] > dist_hi)

ppm_q1, ppm_q3 = df['price_per_mi'].quantile([0.25, 0.75])
ppm_iqr = ppm_q3 - ppm_q1
ppm_lo  = ppm_q1 - 3.0 * ppm_iqr
ppm_hi  = ppm_q3 + 3.0 * ppm_iqr
df['is_pricepmi_outlier'] = (df['price_per_mi'] < ppm_lo) | (df['price_per_mi'] > ppm_hi)

fare_q1, fare_q3 = df['fare'].quantile([0.25, 0.75])
fare_iqr = fare_q3 - fare_q1
fare_lo  = fare_q1 - 3.0 * fare_iqr
fare_hi  = fare_q3 + 3.0 * fare_iqr
df['is_fare_outlier'] = (df['fare'] < fare_lo) | (df['fare'] > fare_hi)

df['is_speed_outlier'] = df['speed_mph'] > 100

# drop same-zone trips where price per mile is also an outlier
df = df[~((df['is_pricepmi_outlier'] == True) & (df['pu_zone_id'] == df['do_zone_id']))]

df.to_parquet(r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data/clean_green.parquet', index=False)

## hvfhv (uber and lyft)

hvfhv is the high-volume for-hire vehicle data - basically uber and lyft. this is by far the biggest dataset so we use duckdb to process it rather than loading everything into pandas memory.

we do the same iqr flagging but in sql, computing the quantiles in a cte and joining them back.

In [ ]:
import duckdb

IN_PATH  = r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data/fhvhv.parquet'
OUT_PATH = r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data/clean_fhvhv.parquet'

con = duckdb.connect()

print('cleaning and writing with duckdb...')

con.execute(f"""
    COPY (
        WITH base AS (
            SELECT
                time_bucket(INTERVAL '30 minutes', pickup_datetime) AS pickup_datetime,
                time_bucket(INTERVAL '30 minutes', dropoff_datetime) AS dropoff_datetime,
                PULocationID AS pu_zone_id,
                DOLocationID AS do_zone_id,
                'hvfhv' AS cab_type,
                trip_miles AS trip_distance_mi,
                trip_time / 60.0 AS trip_duration_min,
                trip_miles / (trip_time / 3600.0) AS speed_mph,
                base_passenger_fare AS fare,
                CASE shared_match_flag WHEN 'Y' THEN true WHEN 'N' THEN false ELSE NULL END AS is_shared,
                base_passenger_fare / trip_miles AS price_per_mi
            FROM read_parquet('{IN_PATH}')
            WHERE PULocationID NOT IN (264, 265)
              AND DOLocationID NOT IN (264, 265)
              AND PULocationID != DOLocationID
              AND trip_time > 0
              AND trip_miles > 0
        ),
        quantiles AS (
            SELECT
                quantile_cont(trip_distance_mi, 0.25) AS dist_q1,
                quantile_cont(trip_distance_mi, 0.75) AS dist_q3,
                quantile_cont(price_per_mi, 0.25)     AS ppm_q1,
                quantile_cont(price_per_mi, 0.75)     AS ppm_q3,
                quantile_cont(fare, 0.25)              AS fare_q1,
                quantile_cont(fare, 0.75)              AS fare_q3
            FROM base
        )
        SELECT
            b.pickup_datetime, b.dropoff_datetime, b.pu_zone_id, b.do_zone_id,
            b.cab_type, b.trip_distance_mi, b.trip_duration_min, b.speed_mph,
            b.fare, b.is_shared, b.price_per_mi,
            (b.trip_distance_mi < (q.dist_q1 - 3.0 * (q.dist_q3 - q.dist_q1)) OR
             b.trip_distance_mi > (q.dist_q3 + 3.0 * (q.dist_q3 - q.dist_q1))) AS is_dist_outlier,
            (b.price_per_mi < (q.ppm_q1 - 3.0 * (q.ppm_q3 - q.ppm_q1)) OR
             b.price_per_mi > (q.ppm_q3 + 3.0 * (q.ppm_q3 - q.ppm_q1))) AS is_pricepmi_outlier,
            (b.fare < (q.fare_q1 - 3.0 * (q.fare_q3 - q.fare_q1)) OR
             b.fare > (q.fare_q3 + 3.0 * (q.fare_q3 - q.fare_q1))) AS is_fare_outlier,
            (b.speed_mph > 100) AS is_speed_outlier
        FROM base b, quantiles q
        WHERE NOT ((b.price_per_mi < (q.ppm_q1 - 3.0 * (q.ppm_q3 - q.ppm_q1)) OR
                    b.price_per_mi > (q.ppm_q3 + 3.0 * (q.ppm_q3 - q.ppm_q1)))
                   AND b.pu_zone_id = b.do_zone_id)
          AND b.pickup_datetime >= '2022-01-01' AND b.pickup_datetime <= '2026-12-31'
          AND b.dropoff_datetime >= '2022-01-01' AND b.dropoff_datetime <= '2026-12-31'
          AND b.trip_distance_mi IS NOT NULL AND b.trip_duration_min IS NOT NULL
          AND b.speed_mph IS NOT NULL AND b.fare IS NOT NULL AND b.price_per_mi IS NOT NULL
    ) TO '{OUT_PATH}' (FORMAT PARQUET)
""")

row_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_PATH}')").fetchone()[0]
print(f'done: {row_count:,} rows -> {OUT_PATH}')

## filter 2024 only

pranav's modelling work uses just 2024 data, so we pull that out as a separate file to keep things manageable.

In [ ]:
import duckdb

IN_PATH  = r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data/clean_fhvhv.parquet'
OUT_PATH = r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data/clean_fhvhv_2024.parquet'

con = duckdb.connect()

con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{IN_PATH}')
        WHERE pickup_datetime >= '2024-01-01'
          AND pickup_datetime <= '2024-12-31'
    ) TO '{OUT_PATH}' (FORMAT PARQUET)
""")

row_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_PATH}')").fetchone()[0]
print(f'done: {row_count:,} rows -> {OUT_PATH}')